# Q-learning

Dans cette démonstration, nous allons étudier une implémentation complète du [Q-learning](https://fr.wikipedia.org/wiki/Q-learning). C'est une méthode basée sur l'estimation de la valeur des paires action-état.

La table qui contient les valeurs estimées des paires action-état est optimisée avec un algorithme proche de l'algorithme de [différence temporelle](https://fr.wikipedia.org/wiki/Temporal_difference_learning), avec la différence notable qu'on optimise $Q(s, a)$ et non pas  $V(s)$ (la récompense attendue en effectuant l'action $a$ dans l'état $s$ et non la valeur globale de l'état $s$).

## Imports

In [ ]:
from collections.abc import Hashable, Sequence
from io import StringIO
from itertools import product
from random import choice as random_choice
from random import random
from typing import Any, Protocol, cast

import gymnasium as gym
from numpy import float32, linspace, zeros

## Types

Ces types sont complètements optionnels, ils sont précisés ici pour informer sur le type des objets ci-dessous.

In [ ]:
class BaseEnvironment[StateT: Hashable, ActionT: Hashable](Protocol):
  def step(
    self, action: ActionT
  ) -> tuple[StateT, float, bool, bool, dict[Any, Any]]: ...

  def reset(self) -> tuple[StateT, dict[str, Any]]: ...

## Q-learning

3 fonctions importantes :

- `bellman_update` implémente la mise à jour par différence temporelle
- `episode` implémente la séquence de `bellman_update`s d'un épisode
- `learn` implémente la séquence d'`episode`s d'un apprentissage complet

In [ ]:
class QLearner[StateT: Hashable, ActionT: Hashable]:
  def __init__(
    self,
    actions: Sequence[ActionT],
    states: Sequence[StateT],
    environment: BaseEnvironment[StateT, ActionT],
    gamma: float = 0.9,
    alpha: float = 0.1,
    n_iter: int = 100,
  ):
    self.actions = actions
    self.action_to_index = {a: i for i, a in enumerate(actions)}
    self.states = states
    self.state_to_index = {s: i for i, s in enumerate(states)}
    self.environment = environment
    self.gamma = gamma
    self.alpha = alpha
    self.n_iter = n_iter
    self.q_table = zeros((len(states), len(actions)), dtype=float32)

  def __str__(self) -> str:
    with StringIO() as string_io:
      actions_str = " | ".join(f"{a!s:^5}" for a in self.actions)
      string_io.write(f"{' ' * 10} | {actions_str}\n")
      for state, row in zip(self.states, self.q_table):
        state_values = " | ".join(f"{v:5.2f}" for v in row)
        string_io.write(f"{state!s:10} | {state_values}\n")
      return string_io.getvalue()

  def learn(self) -> float:
    len_i = len(str(self.n_iter))
    for i, epsilon in enumerate(linspace(1, 0, self.n_iter), start=1):
      self.epsilon = epsilon
      reward = self.episode()
      print(f"iteration {i:{len_i}}, ε = {epsilon:.2f}, r = {reward:7.2f}")
    return reward

  def get_q(self, state: StateT, action: ActionT) -> float:
    return self.q_table[self.state_to_index[state], self.action_to_index[action]]

  def set_q(self, state: StateT, action: ActionT, value: float) -> None:
    self.q_table[self.state_to_index[state], self.action_to_index[action]] = value

  def bellman_update(self, s: StateT, a: ActionT, r: float, new_s: StateT) -> None:
    q = self.get_q(s, a)
    best_new_s_a = self.best_action(new_s)
    new_s_max_q = self.get_q(new_s, best_new_s_a)
    self.set_q(s, a, q + self.alpha * (r + self.gamma * new_s_max_q - q))

  def best_action(self, s: StateT) -> ActionT:
    _, a = max((self.get_q(s, a), a) for a in self.actions)
    return a

  def episode(self) -> float:
    s, _ = self.environment.reset()
    total_r = 0.0
    done = False
    while not done:
      if random() >= self.epsilon:
        # Exploitation
        a = self.best_action(s)
      else:
        # Exploration
        a = random_choice(self.actions)
      new_s, r, done, _, _ = self.environment.step(a)
      total_r += r
      self.bellman_update(s, a, r, new_s)
      s = new_s
    return total_r

## Utilisation

Pour utiliser un algorithme de Q-learning, il faut définir un environnement, en particulier sa fonction `step`, qui récupère l'action de l'agent et rend le nouvel état, la récompense, et la terminaison de l'épisode.

In [ ]:
class Environment:
  def __init__(self) -> None:
    self.allowed = frozenset(
      {
        (0, 1),
        (0, 3),
        (0, 4),
        (0, 5),
        (1, 1),
        (1, 3),
        (2, 1),
        (2, 2),
        (2, 3),
        (2, 4),
        (2, 5),
        (2, 6),
        (3, 1),
        (4, 1),
      }
    )

  def reset(self) -> tuple[tuple[int, int], dict[str, Any]]:
    self.current_state = (2, 0)
    return self.current_state, {}

  def step(
    self, action: str
  ) -> tuple[tuple[int, int], float, bool, bool, dict[Any, Any]]:
    r, c = self.current_state
    if action == "↑":
      r -= 1
    elif action == "→":
      c += 1
    elif action == "↓":
      r += 1
    elif action == "←":
      c -= 1
    else:
      raise ValueError("Invalid action")
    if 0 <= r < 5 and 1 <= c < 6 and (r, c) in self.allowed:
      self.current_state = (r, c)
      return (r, c), -1.0, False, False, {}
    if (r, c) == (2, 6):
      self.current_state = (r, c)
      return (r, c), -1.0, True, False, {}
    return self.current_state, -1.0, False, False, {}

In [ ]:
def test_qlearn() -> None:
  actions = ["↑", "→", "↓", "←"]
  states = [(i, j) for i, j in product(range(5), range(1, 6))] + [(2, 0), (2, 6)]

  environment = Environment()

  q_learner: QLearner[tuple[int, int], str] = QLearner(
    actions=actions, states=states, environment=environment, n_iter=1000
  )
  final_reward = q_learner.learn()
  assert int(final_reward) == -6
  print(q_learner)

In [ ]:
test_qlearn()

In [ ]:
def test_frozen() -> None:
  env = cast("BaseEnvironment[int, int]", gym.make("FrozenLake-v1"))
  q_learner = QLearner(
    actions=[0, 1, 2, 3], states=list(range(16)), environment=env, n_iter=10000
  )
  final_reward = q_learner.learn()
  print(final_reward, q_learner)

In [ ]:
test_frozen()